In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
from sqlalchemy import create_engine, text
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set pandas display options
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

### loads `.env`
#### Setting up the `database connection`

In [2]:
load_dotenv()

pg_url = (
    f"postgresql+psycopg2://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT')}/{os.getenv('PGDATABASE')}"
)

engine = create_engine(pg_url, pool_pre_ping=True)

with engine.begin() as conn:
    conn.execute(text("SET search_path TO mart, curated, public;"))

with engine.begin() as conn:
    print(conn.execute(text("SELECT now()")).scalar())

2025-10-24 17:15:29.409791-04:00


In [4]:
# Pull claim features + flags, built in earlier steps
q_claims = """
SELECT claimid, provider, beneid, claim_type, claim_start, claim_end,
       reimb_amt, deductible_paid, los_days, dx_count, px_count, drg
FROM mart.features_claim
"""
claims = pd.read_sql(q_claims, con=engine)

# Rule flags (duplicates / upcoding / overcharge) — from 2.4
rcf = pd.read_sql("SELECT * FROM mart.rule_claim_flags", con=engine)

# Claim-level z-scores (built in 2.4) for precise peer-normalized checks
z = pd.read_sql(
    "SELECT claimid, z_ip_reimb, z_ip_los, z_op_reimb FROM mart.rule_claim_z", con=engine)

df = (claims
      .merge(rcf, on="claimid", how="left")
      .merge(z, on="claimid", how="left"))

df

,claimid,provider,beneid,claim_type,claim_start,claim_end,reimb_amt,deductible_paid,los_days,dx_count,...,drg,dup_exact_flag,dup_near_count,upcoding_ip_flag,upcoding_op_flag,overcharge_z_flag,overcharge_iqr_flag,z_ip_reimb,z_ip_los,z_op_reimb
0,CLM569367,PRV55455,BENE100014,OP,2009-09-08,2009-09-08,100.00,0.00,NaN,1,...,None,0,0,0,0,0,0,NaN,NaN,-0.30
1,CLM76080,PRV55659,BENE100014,IP,2009-11-15,2009-11-17,"3,000.00","1,068.00",2.00,6,...,181,0,0,0,0,0,0,-0.67,-0.67,NaN
2,CLM280761,PRV55832,BENE100016,OP,2009-04-02,2009-04-02,60.00,0.00,NaN,2,...,None,0,0,0,0,0,0,NaN,NaN,-0.35
3,CLM174738,PRV55368,BENE100021,OP,2009-02-03,2009-02-03,50.00,0.00,NaN,2,...,None,0,0,0,0,0,0,NaN,NaN,-0.38
4,CLM296629,PRV55209,BENE100040,OP,2009-04-10,2009-04-10,200.00,0.00,NaN,2,...,None,0,0,0,0,0,0,NaN,NaN,-0.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
558206,CLM195970,PRV51074,BENE99980,OP,2009-02-15,2009-02-15,40.00,0.00,NaN,1,...,None,0,0,0,0,0,0,NaN,NaN,-0.50
558207,CLM693942,PRV51117,BENE99980,OP,2009-11-22,2009-11-22,80.00,0.00,NaN,2,...,None,0,0,0,0,0,0,NaN,NaN,-0.30
558208,CLM491509,PRV57605,BENE99981,OP,2009-07-26,2009-07-26,400.00,0.00,NaN,5,...,None,0,0,0,0,0,0,NaN,NaN,0.32
558209,CLM117724,PRV56218,BENE99987,OP,2009-01-03,2009-01-03,100.00,0.00,NaN,1,...,None,0,0,0,0,0,0,NaN,NaN,-0.30


In [6]:
def iqr_flags(s: pd.Series):
    x = s.dropna()
    q1, q3 = x.quantile([.25, .75])
    iqr = q3 - q1
    return (s < (q1 - 1.5*iqr)) | (s > (q3 + 1.5*iqr))


def modified_z_flags(s: pd.Series):
    x = s.dropna()
    med = x.median()
    mad = (x - med).abs().median()
    if mad == 0 or np.isnan(mad):
        return pd.Series(False, index=s.index)
    Mi = 0.6745 * (s - med) / mad
    return Mi.abs() > 3.5

In [7]:
df["iqr_outlier_reimb"] = df.groupby(
    "claim_type")["reimb_amt"].transform(iqr_flags)
df["mad_outlier_reimb"] = df.groupby(
    "claim_type")["reimb_amt"].transform(modified_z_flags)

#### 2 - `Duplicate` & `near-duplicate` claims

In [8]:
df["duplicate_suspect"] = (df["dup_exact_flag"].fillna(
    0).eq(1)) | (df["dup_near_count"].fillna(0) > 0)

##### 3 - `Short-stay`, `high-payment` inpatient claims

In [9]:
df["short_stay_flag_py"] = (df["claim_type"].eq(
    "IP")) & (df["los_days"].fillna(0) < 2)
df["short_stay_costly"] = df["short_stay_flag_py"] & (
    df["z_ip_reimb"].fillna(0) > 2.5)

##### 4 - `Missingness` & `odd` shapes

In [10]:
# Key-field nulls
null_checks = {
    "reimb_amt_null": df["reimb_amt"].isna().sum(),
    "ip_missing_drg": df.query("claim_type=='IP'")["drg"].isna().sum(),
    "ip_missing_dates": df.query("claim_type=='IP'")[["claim_start", "claim_end"]].isna().any(axis=1).sum(),
    "op_missing_dxcount": df.query("claim_type=='OP'")["dx_count"].isna().sum()
}
null_checks

{'reimb_amt_null': np.int64(0),
 'ip_missing_drg': np.int64(0),
 'ip_missing_dates': np.int64(0),
 'op_missing_dxcount': np.int64(0)}

##### 5 - Build `case packets`

In [12]:
cols = ["claimid", "provider", "beneid", "claim_type", "claim_start", "claim_end",
        "reimb_amt", "deductible_paid", "los_days", "dx_count", "px_count", "drg",
        "dup_exact_flag", "dup_near_count", "z_ip_reimb", "z_ip_los", "z_op_reimb"]

pkt_outlier = df.loc[df["iqr_outlier_reimb"]
                     | df["mad_outlier_reimb"], cols].copy()
pkt_dupe = df.loc[df["duplicate_suspect"], cols].copy()
pkt_short = df.loc[df["short_stay_costly"], cols].copy()

# A “top spend” slice: largest paid in top 1% by type
cut = df.groupby("claim_type")["reimb_amt"].transform(
    lambda s: s.quantile(0.99))
pkt_toppaid = df.loc[df["reimb_amt"] >= cut, cols].copy()

# Combine and add a reason tag


def tag(df_, label):
    x = df_.copy()
    x["reason"] = label
    return x


case_packets = pd.concat([
    tag(pkt_outlier, "outlier_reimb_iqr_or_mad"),
    tag(pkt_dupe,    "duplicate_or_near_duplicate"),
    tag(pkt_short,   "short_stay_high_zpay_ip"),
    tag(pkt_toppaid, "top1pct_paid")
], ignore_index=True).drop_duplicates(subset=["claimid", "reason"])
case_packets.head(100)

,claimid,provider,beneid,claim_type,claim_start,claim_end,reimb_amt,deductible_paid,los_days,dx_count,px_count,drg,dup_exact_flag,dup_near_count,z_ip_reimb,z_ip_los,z_op_reimb,reason
0,CLM434931,PRV53005,BENE100146,OP,2009-06-25,2009-06-25,500.00,0.00,NaN,4,0,None,0,0,NaN,NaN,0.07,outlier_reimb_iqr_or_mad
1,CLM416648,PRV57434,BENE100154,OP,2009-06-15,2009-06-15,800.00,0.00,NaN,4,0,None,0,0,NaN,NaN,1.23,outlier_reimb_iqr_or_mad
2,CLM285505,PRV51860,BENE100180,OP,2009-04-04,2009-04-04,"1,300.00",0.00,NaN,3,0,None,0,0,NaN,NaN,1.81,outlier_reimb_iqr_or_mad
3,CLM234546,PRV55834,BENE100264,OP,2009-03-08,2009-03-08,"2,000.00",0.00,NaN,0,0,None,0,0,NaN,NaN,NaN,outlier_reimb_iqr_or_mad
4,CLM119588,PRV57191,BENE100299,OP,2009-01-04,2009-01-04,"1,200.00",0.00,NaN,3,0,None,0,0,NaN,NaN,1.62,outlier_reimb_iqr_or_mad
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CLM242895,PRV51250,BENE103859,OP,2009-03-12,2009-03-12,"2,300.00",0.00,NaN,1,0,None,0,0,NaN,NaN,3.00,outlier_reimb_iqr_or_mad
96,CLM724144,PRV56738,BENE103864,OP,2009-12-11,2009-12-11,800.00,0.00,NaN,4,0,None,0,0,NaN,NaN,1.23,outlier_reimb_iqr_or_mad
97,CLM137806,PRV54367,BENE103882,OP,2009-01-14,2009-01-14,"1,800.00",0.00,NaN,1,0,None,0,0,NaN,NaN,1.82,outlier_reimb_iqr_or_mad
98,CLM244753,PRV53773,BENE103901,OP,2009-03-13,2009-04-02,400.00,0.00,NaN,10,0,None,0,0,NaN,NaN,0.07,outlier_reimb_iqr_or_mad


##### Quick sanity counts

In [13]:
summary = pd.DataFrame({
    "outlier_reimb_iqr_or_mad": [len(pkt_outlier)],
    "duplicate_or_near_duplicate": [len(pkt_dupe)],
    "short_stay_high_zpay_ip": [len(pkt_short)],
    "top1pct_paid": [len(pkt_toppaid)],
})
summary.T.rename(columns={0: "n_claims"})

,n_claims
outlier_reimb_iqr_or_mad,101896
duplicate_or_near_duplicate,583
short_stay_high_zpay_ip,82
top1pct_paid,6439


##### Save packets for review

In [14]:
case_packets.to_csv("case_packets_claims.csv", index=False)

##### Derive the boolean flags

In [15]:
# Top-1% paid within claim_type
q99 = df.groupby("claim_type")["reimb_amt"].transform(lambda s: s.quantile(0.99))
df["is_top1pct_paid"] = df["reimb_amt"] >= q99  # boolean

# High peer z (either IP or OP)
df["peer_z_high"] = (
    (df["z_ip_reimb"].fillna(-np.inf) > 3) |
    (df["z_op_reimb"].fillna(-np.inf) > 3)
)

# Unified outlier flag from IQR/MAD (created in 3.4)
df["is_outlier_reimb"] = df["iqr_outlier_reimb"] | df["mad_outlier_reimb"]

# Duplicate suspect (created in 3.4)
df["duplicate_suspect"] = df["duplicate_suspect"].fillna(False)

# Overcharge rule flags from 2.4 (if present)
for col in ["overcharge_z_flag","overcharge_iqr_flag","dup_exact_flag","dup_near_count"]:
    if col in df.columns:
        df[col] = df[col].fillna(0).astype(int)

# Helpful view of counts (sanity)
df[["is_top1pct_paid","peer_z_high","is_outlier_reimb","duplicate_suspect"]].sum()

,claimid,provider,beneid,claim_type,claim_start,claim_end,reimb_amt,deductible_paid,los_days,dx_count,...,z_ip_los,z_op_reimb,iqr_outlier_reimb,mad_outlier_reimb,duplicate_suspect,short_stay_flag_py,short_stay_costly,is_top1pct_paid,peer_z_high,is_outlier_reimb
0,CLM569367,PRV55455,BENE100014,OP,2009-09-08,2009-09-08,100.00,0.00,NaN,1,...,NaN,-0.30,False,False,False,False,False,False,False,False
1,CLM76080,PRV55659,BENE100014,IP,2009-11-15,2009-11-17,"3,000.00","1,068.00",2.00,6,...,-0.67,NaN,False,False,False,False,False,False,False,False
2,CLM280761,PRV55832,BENE100016,OP,2009-04-02,2009-04-02,60.00,0.00,NaN,2,...,NaN,-0.35,False,False,False,False,False,False,False,False
3,CLM174738,PRV55368,BENE100021,OP,2009-02-03,2009-02-03,50.00,0.00,NaN,2,...,NaN,-0.38,False,False,False,False,False,False,False,False
4,CLM296629,PRV55209,BENE100040,OP,2009-04-10,2009-04-10,200.00,0.00,NaN,2,...,NaN,-0.03,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
558206,CLM195970,PRV51074,BENE99980,OP,2009-02-15,2009-02-15,40.00,0.00,NaN,1,...,NaN,-0.50,False,False,False,False,False,False,False,False
558207,CLM693942,PRV51117,BENE99980,OP,2009-11-22,2009-11-22,80.00,0.00,NaN,2,...,NaN,-0.30,False,False,False,False,False,False,False,False
558208,CLM491509,PRV57605,BENE99981,OP,2009-07-26,2009-07-26,400.00,0.00,NaN,5,...,NaN,0.32,False,True,False,False,False,False,False,True
558209,CLM117724,PRV56218,BENE99987,OP,2009-01-03,2009-01-03,100.00,0.00,NaN,1,...,NaN,-0.30,False,False,False,False,False,False,False,False


##### Build intersection slices (claim-level “case packets”)

In [24]:
cols_core = ["claimid", "provider", "beneid", "claim_type", "claim_start", "claim_end",
             "reimb_amt", "deductible_paid", "los_days", "dx_count", "px_count", "drg",
             "z_ip_reimb", "z_op_reimb", "z_ip_los",
             "dup_exact_flag", "dup_near_count", "overcharge_z_flag", "overcharge_iqr_flag"]


def pick(cols):
    return [c for c in cols if c in df.columns]


pkt_short_and_top = df.loc[df["short_stay_costly"]
                           & df["is_top1pct_paid"], pick(cols_core)].copy()
pkt_dup_and_peer = df.loc[df["duplicate_suspect"] &
                          df["peer_z_high"],     pick(cols_core)].copy()
pkt_outlier_and_top = df.loc[df["is_outlier_reimb"]
                             & df["is_top1pct_paid"],  pick(cols_core)].copy()
pkt_overchg_and_top = df.loc[(df.get("overcharge_z_flag", 0).eq(1) | df.get(
    "overcharge_iqr_flag", 0).eq(1)) & df["is_top1pct_paid"], pick(cols_core)].copy()

# Label each packet with a reason (for triage)


def tag(d, label):
    x = d.copy()
    x["reason"] = label
    return x


claims_queue = pd.concat([
    tag(pkt_short_and_top,  "short_stay_high_zpay_ip ∩ top1pct"),
    tag(pkt_dup_and_peer,   "duplicate_suspect ∩ peer_z>3"),
    tag(pkt_outlier_and_top, "outlier(IQR|MAD) ∩ top1pct"),
    tag(pkt_overchg_and_top, "overcharge_rule ∩ top1pct")
], ignore_index=True).drop_duplicates(subset=["claimid", "reason"])

# Rank inside each reason by dollars (largest first)
claims_queue["rank_in_reason"] = claims_queue.groupby("reason")["reimb_amt"].rank(
    method="first", ascending=False)  # percentile ranks are below

claims_queue.head(100)

,claimid,provider,beneid,claim_type,claim_start,claim_end,reimb_amt,deductible_paid,los_days,dx_count,...,drg,z_ip_reimb,z_op_reimb,z_ip_los,dup_exact_flag,dup_near_count,overcharge_z_flag,overcharge_iqr_flag,reason,rank_in_reason
0,CLM69520,PRV52151,BENE74032,IP,2009-09-25,2009-09-26,"64,000.00","1,068.00",1.00,5,...,220,3.00,NaN,-0.82,0,0,0,1,short_stay_high_zpay_ip ∩ top1pct,6.00
1,CLM72294,PRV54504,BENE64896,IP,2009-10-17,2009-10-18,"57,000.00","1,068.00",1.00,9,...,294,3.00,NaN,-0.72,0,0,0,1,short_stay_high_zpay_ip ∩ top1pct,9.00
2,CLM78738,PRV51274,BENE131509,IP,2009-12-07,2009-12-08,"59,000.00","1,068.00",1.00,5,...,466,3.00,NaN,-1.09,0,0,0,1,short_stay_high_zpay_ip ∩ top1pct,8.00
3,CLM80589,PRV57103,BENE147111,IP,2009-12-23,2009-12-24,"57,000.00","1,068.00",1.00,9,...,292,3.00,NaN,-0.84,0,0,1,1,short_stay_high_zpay_ip ∩ top1pct,10.00
4,CLM51986,PRV53033,BENE69570,IP,2009-05-20,2009-05-21,"73,000.00","1,068.00",1.00,7,...,424,3.00,NaN,-0.88,0,0,0,1,short_stay_high_zpay_ip ∩ top1pct,4.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,CLM669050,PRV51574,BENE153909,OP,2009-11-06,2009-11-06,"3,300.00",0.00,NaN,1,...,None,NaN,3.00,NaN,0,0,0,1,outlier(IQR|MAD) ∩ top1pct,"1,133.00"
96,CLM712674,PRV52120,BENE153921,OP,2009-12-03,2009-12-03,"7,700.00",0.00,NaN,1,...,None,NaN,3.00,NaN,0,0,0,1,outlier(IQR|MAD) ∩ top1pct,683.00
97,CLM608343,PRV57157,BENE15415,OP,2009-10-01,2009-10-01,"3,300.00",0.00,NaN,9,...,None,NaN,3.00,NaN,0,0,1,1,outlier(IQR|MAD) ∩ top1pct,"1,134.00"
98,CLM292311,PRV51347,BENE154447,OP,2009-04-08,2009-04-08,"3,300.00",0.00,NaN,3,...,None,NaN,3.00,NaN,0,0,1,1,outlier(IQR|MAD) ∩ top1pct,"1,135.00"


##### Provider-level triage table (counts, dollars, composite score)

In [22]:
# Build provider-level signals from the intersection claims
prov_agg = (claims_queue
            .groupby(["provider","reason"])
            .agg(n_claims=("claimid","count"),
                 dollars=("reimb_amt","sum"),
                 max_z_ip=("z_ip_reimb","max"),
                 max_z_op=("z_op_reimb","max"),
                 min_los=("z_ip_los","min"))   # min z_los => shorter than peers
            .reset_index())

# Pivot reasons to columns (counts + dollars per reason)
prov_pivot_n = prov_agg.pivot_table(index="provider", columns="reason", values="n_claims", fill_value=0, aggfunc="sum")
prov_pivot = prov_agg.pivot_table(index="provider", columns="reason", values="dollars",  fill_value=0.0, aggfunc="sum")

# Flatten MultiIndex columns
prov_pivot_n.columns = [f"n__{c}" for c in prov_pivot_n.columns]
prov_pivot.columns = [f"$$__{c}" for c in prov_pivot.columns]

prov_sum = (prov_pivot_n.join(prov_pivot, how="outer")
            .fillna(0)
            .reset_index())

# Add global totals and a simple composite score (counts + normalized dollars)
count_cols = [c for c in prov_sum.columns if c.startswith("n__")]
dollar_cols= [c for c in prov_sum.columns if c.startswith("$$__")]

prov_sum["n_any"] = prov_sum[count_cols].sum(axis=1)
prov_sum["dollars_any"] = prov_sum[dollar_cols].sum(axis=1)

# Percentile ranks (0..1) per column for a simple “risk_score”
for c in count_cols + dollar_cols:
    prov_sum[f"r_{c}"] = prov_sum[c].rank(pct=True, ascending=False)

rank_cols = [c for c in prov_sum.columns if c.startswith("r_")]
prov_sum["risk_score_intersections"] = prov_sum[rank_cols].mean(axis=1)  # equal weights

# Order your audit queue by score, break ties by dollars then count
audit_queue_providers = prov_sum.sort_values(
    ["risk_score_intersections","dollars_any","n_any"], ascending=[False, False, False]
)

audit_queue_providers.head(20)


,provider,n__duplicate_suspect ∩ peer_z>3,n__outlier(IQR|MAD) ∩ top1pct,n__overcharge_rule ∩ top1pct,n__short_stay_high_zpay_ip ∩ top1pct,$$__duplicate_suspect ∩ peer_z>3,$$__outlier(IQR|MAD) ∩ top1pct,$$__overcharge_rule ∩ top1pct,$$__short_stay_high_zpay_ip ∩ top1pct,n_any,dollars_any,r_n__duplicate_suspect ∩ peer_z>3,r_n__outlier(IQR|MAD) ∩ top1pct,r_n__overcharge_rule ∩ top1pct,r_n__short_stay_high_zpay_ip ∩ top1pct,r_$$__duplicate_suspect ∩ peer_z>3,r_$$__outlier(IQR|MAD) ∩ top1pct,r_$$__overcharge_rule ∩ top1pct,r_$$__short_stay_high_zpay_ip ∩ top1pct,risk_score_intersections
100,PRV51277,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
480,PRV52439,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
493,PRV52487,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
618,PRV52905,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
695,PRV53222,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
739,PRV53362,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
749,PRV53388,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
1105,PRV54681,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
1289,PRV55187,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
1421,PRV55513,0,1,0,0,0.00,"3,300.00",0.00,0.00,1,"3,300.00",0.50,0.76,0.99,0.50,0.50,0.81,0.99,0.50,0.69
